<a href="https://colab.research.google.com/github/suneel-ratnala/Agentic_AI_Practice/blob/main/Exercises%5CReAct_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install Dependencies
print("Installing dependencies...")
%pip install -qU langchain-groq langchain-community langgraph langsmith langchain_core langchain-groq
print("Dependencies installed.")

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.1/359.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langchain 1.2.10 requires langgraph<1.1.0,>=1.0.8, but you have langg

In [2]:
# 2. Setup API Keys
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [5]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import ToolMessage # Passes data back to LLM after it calls a tool such as the content and the tool_call_id
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


@tool
def add(a: int, b:int):
    """This is an addition function that adds 2 numbers together"""
    return a + b

@tool
def subtract(a: int, b: int):
    """Subtraction function"""
    return a - b

@tool
def multiply(a: int, b: int):
    """Multiplication function"""
    return a * b

tools = [add, subtract, multiply]

# model = ChatOpenAI(model = "gpt-4o").bind_tools(tools)
from langchain_core.messages import SystemMessage

# Initialize the LLM (Large Language Model)
# We are using Groq for fast inference.
print("Initializing LLM...")
model = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
).bind_tools(tools)

def model_call(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content=
        "You are my AI assistant, please answer my query to the best of your ability."
    )
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    print(last_message.tool_calls)
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"


graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)


tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.set_entry_point("our_agent")

graph.add_conditional_edges(
    "our_agent",
    should_continue,
    {
        "continue": "tools",
        "end": END,
    },
)

graph.add_edge("tools", "our_agent")

app = graph.compile()

def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [("user", "Add 40 + 12 and then multiply the result by 6. Also tell me a joke please.")]}
#inputs = {"messages": [("user", "Add 40 + 12")]}
print_stream(app.stream(inputs, stream_mode="values"))

Initializing LLM...
================================ Human Message =================================

Add 40 + 12 and then multiply the result by 6. Also tell me a joke please.
[{'name': 'add', 'args': {'a': 40, 'b': 12}, 'id': '54ndapz64', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 52, 'b': 6}, 'id': 'cyf9tjbc0', 'type': 'tool_call'}]
================================== Ai Message ==================================
Tool Calls:
  add (54ndapz64)
 Call ID: 54ndapz64
  Args:
    a: 40
    b: 12
  multiply (cyf9tjbc0)
 Call ID: cyf9tjbc0
  Args:
    a: 52
    b: 6
================================= Tool Message =================================
Name: multiply

312
[]
================================== Ai Message ==================================

The result of adding 40 and 12 is **52**, and multiplying that by 6 gives **312**.  

Here's a joke for you:  
**"Why did the math book look sad? Because it had too many problems."** 😄


In [6]:
app.invoke(inputs)
print("Done!")

[{'name': 'add', 'args': {'a': 40, 'b': 12}, 'id': 'qj344cym0', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 52, 'b': 6}, 'id': 'f7z6qxvkz', 'type': 'tool_call'}]
[]
Done!
